<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_28_practicum_data_structures/note_lesson_28_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧱 Урок 28 — Практикум П6: структури даних диспетчерської → пакет `dispatch`

П'ять питань диспетчерської «Смачно + Таксі» — п'ять структур:

| Питання | Структура | Вправа |
|---|---|---|
| скасувати останнє призначення | стек `Stack` | 1–2 |
| хто прийшов першим | черга `Queue` | 3 |
| найтерміновіше замовлення | купа `MinHeap` | 4 |
| автодоповнення адрес | префіксне дерево `Trie` | 5 |
| ціни маршрутів без повторних запитів | LRU-кеш `LRUCache` | 6 |

Потім — готовий пакет `dispatch`, у якому ці структури працюють разом, і його 57 тестів (вправа 7).

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Теорія й покрокові схеми — у книзі: [Урок 28](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_28/).

In [ ]:
import heapq
import random
import time
from collections import OrderedDict, deque


class EmptyError(LookupError):
    """Спроба взяти елемент з порожньої структури."""

---
## 1. Стек: скасувати останнє

LIFO: останнім поклали — першим узяли. Вершина — **кінець** списку: там `append` і `pop()` коштують `O(1)`.

### Вправа 1. Клас `Stack`

Методи `push`, `pop`, `peek` (`pop` і `peek` на порожньому стеку кидають `EmptyError`), а також `__len__`, `__bool__` і `__iter__` — від вершини до дна.

In [ ]:
class Stack:
    def __init__(self, items=()):
        self._items = list(items)

    # YOUR CODE HERE
    # BEGIN SOLUTION
    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise EmptyError("стек порожній")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise EmptyError("стек порожній")
        return self._items[-1]

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __iter__(self):
        return reversed(self._items)
    # END SOLUTION

    def __repr__(self):
        return f"Stack({self._items!r})"


history = Stack()
for action in [("Оксана", 104), ("Тарас", 103), ("Ігор", 101)]:
    history.push(action)
print(list(history))
assert history.pop() == ("Ігор", 101)
assert history.peek() == ("Тарас", 103) and len(history) == 2
assert list(Stack([1, 2, 3])) == [3, 2, 1]
assert not Stack()
for take in (Stack.pop, Stack.peek):
    try:
        take(Stack())
        raise AssertionError("порожній стек мав кинути EmptyError")
    except EmptyError:
        pass
print("✅ Вправа 1 пройдена")

### Вправа 2. Перевірка шаблону SMS

`is_balanced(text)`: кожна дужка `(`, `[`, `{` закрита відповідною і в правильному порядку. Відкривну — у стек; закривна має відповідати **верхній** відкривній; наприкінці стек порожній.

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def is_balanced(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    opened = Stack()
    for char in text:
        if char in "([{":
            opened.push(char)
        elif char in PAIRS:
            if not opened or opened.pop() != PAIRS[char]:
                return False
    return not opened
    # END SOLUTION


assert is_balanced("Замовлення {id} (кур'єр: {name}) [до {time}]")
assert not is_balanced("({)}")
assert not is_balanced("((")
assert not is_balanced("())")
assert is_balanced("без дужок")
print("✅ Вправа 2 пройдена")

---
## 3. Черга: хто перший прийшов

**🔮 Прогноз:** `list.pop(0)` для 50 000 елементів займає близько 0,2 с. Скільки займе для 100 000? А `deque.popleft()`?

<details>
<summary>Відповідь</summary>

`list.pop(0)` — приблизно **учетверо** довше: кожен виклик зсуває всі елементи (`O(n)`), а викликів теж удвічі більше. `deque.popleft()` — `O(1)`, тож лише **удвічі** довше, і в сотні разів швидше за список.

</details>

In [ ]:
for n in (50_000, 100_000):
    items = list(range(n))
    start = time.perf_counter()
    while items:
        items.pop(0)
    slow = time.perf_counter() - start

    queue = deque(range(n))
    start = time.perf_counter()
    while queue:
        queue.popleft()
    fast = time.perf_counter() - start
    print(f"{n:>7}: list.pop(0) {slow:.2f} с, deque.popleft() {fast:.3f} с")

### Вправа 3. Клас `Queue` на `deque`

`enqueue` — у хвіст, `dequeue` — з голови (на порожній — `EmptyError`), `__len__`, `__bool__`, `__iter__` у порядку обслуговування.

In [ ]:
class Queue:
    def __init__(self, items=()):
        self._items = deque(items)

    # YOUR CODE HERE
    # BEGIN SOLUTION
    def enqueue(self, item):
        self._items.append(item)

    def dequeue(self):
        if not self._items:
            raise EmptyError("черга порожня")
        return self._items.popleft()

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __iter__(self):
        return iter(self._items)
    # END SOLUTION

    def __repr__(self):
        return f"Queue({list(self._items)!r})"


orders = Queue()
for number in [101, 102, 105]:
    orders.enqueue(number)
assert orders.dequeue() == 101
assert list(orders) == [102, 105] and len(orders) == 2
try:
    Queue().dequeue()
    raise AssertionError("порожня черга мала кинути EmptyError")
except EmptyError:
    pass
print("✅ Вправа 3 пройдена")

---
## 4. Купа: найтерміновіше замовлення

Бінарне дерево в списку: діти вузла `i` — `2*i + 1` і `2*i + 2`, батько — `(i - 1) // 2`. Правило: батько не більший за дітей, тож мінімум — у `items[0]`.

**🔮 Прогноз:** список-купа `["09:40", "10:05"]`. Додаємо `"09:25"`. Де він опиниться і яким стане список?

<details>
<summary>Відповідь</summary>

Спершу в кінці (індекс 2). Його батько — індекс `(2 - 1) // 2 = 0`, `"09:40"`. `"09:25" < "09:40"` — обмін. Індекс 0 — вершина, стоп: `["09:25", "10:05", "09:40"]`.

</details>

### Вправа 4. `_sift_up` і `_sift_down`

Решту `MinHeap` написано. Допиши два методи:

- `_sift_up(i)` — поки `i > 0` і елемент менший за батька, міняй їх місцями й піднімайся;
- `_sift_down(i)` — знайди меншу з дітей; якщо вона менша за елемент, обміняй і опускайся далі; інакше стоп.

Порівнюй через `self._less(a, b)` — він враховує `key`.

In [ ]:
class MinHeap:
    def __init__(self, items=(), key=None):
        self._key = key if key is not None else (lambda item: item)
        self._items = []
        for item in items:
            self.push(item)

    def push(self, item):
        self._items.append(item)
        self._sift_up(len(self._items) - 1)

    def pop(self):
        if not self._items:
            raise EmptyError("купа порожня")
        top = self._items[0]
        last = self._items.pop()
        if self._items:
            self._items[0] = last
            self._sift_down(0)
        return top

    def _less(self, i, j):
        return self._key(self._items[i]) < self._key(self._items[j])

    def _sift_up(self, i):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        while i > 0:
            parent = (i - 1) // 2
            if not self._less(i, parent):
                break
            self._items[i], self._items[parent] = self._items[parent], self._items[i]
            i = parent
        # END SOLUTION

    def _sift_down(self, i):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        n = len(self._items)
        while True:
            smallest = i
            for child in (2 * i + 1, 2 * i + 2):
                if child < n and self._less(child, smallest):
                    smallest = child
            if smallest == i:
                break
            self._items[i], self._items[smallest] = self._items[smallest], self._items[i]
            i = smallest
        # END SOLUTION

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __repr__(self):
        return f"MinHeap({self._items!r})"


heap = MinHeap()
for deadline in ["09:40", "10:05", "09:25"]:
    heap.push(deadline)
print(heap)
assert heap._items == ["09:25", "10:05", "09:40"]

rng = random.Random(28)
for trial in range(300):
    items = [rng.randint(0, 50) for _ in range(rng.randint(0, 40))]
    h = MinHeap(items)
    std = list(items)
    heapq.heapify(std)
    assert [h.pop() for _ in range(len(h))] == sorted(items) == [heapq.heappop(std) for _ in range(len(std))]
print("✅ Вправа 4 пройдена: 300 випадкових наборів збігаються з sorted і heapq")

---
## 5. Префіксне дерево: автодоповнення

Кожен вузол — словник `літера → вузол` і прапорець `is_word` («тут закінчується ціле слово»).

### Вправа 5. `add`, `__contains__`, `starts_with`

- `add(word)` — пройти по літерах, створюючи відсутні вузли (`setdefault`), позначити останній `is_word = True`;
- `word in trie` — лише цілі слова, не префікси;
- `starts_with(prefix)` — усі слова з префіксом за алфавітом; обхід уже написано в `_collect`.

In [ ]:
class _Node:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children = {}
        self.is_word = False


class Trie:
    def __init__(self, words=()):
        self._root = _Node()
        for word in words:
            self.add(word)

    def _find(self, prefix):
        node = self._root
        for letter in prefix:
            node = node.children.get(letter)
            if node is None:
                return None
        return node

    def _collect(self, node, path, words):
        if node.is_word:
            words.append(path)
        for letter in sorted(node.children):
            self._collect(node.children[letter], path + letter, words)

    # YOUR CODE HERE
    # BEGIN SOLUTION
    def add(self, word):
        node = self._root
        for letter in word:
            node = node.children.setdefault(letter, _Node())
        node.is_word = True

    def __contains__(self, word):
        node = self._find(word)
        return node is not None and node.is_word

    def starts_with(self, prefix):
        node = self._find(prefix)
        if node is None:
            return []
        words = []
        self._collect(node, prefix, words)
        return words
    # END SOLUTION


addresses = Trie(["Хрещатик, 1", "Хрещатик, 22", "Хорива, 5", "Оболонська, 12"])
print(addresses.starts_with("Хр"))
assert addresses.starts_with("Хр") == ["Хрещатик, 1", "Хрещатик, 22"]
assert addresses.starts_with("Х") == ["Хорива, 5", "Хрещатик, 1", "Хрещатик, 22"]
assert addresses.starts_with("Ки") == []
assert "Хорива, 5" in addresses and "Хорива" not in addresses
print("✅ Вправа 5 пройдена")

---
## 6. LRU-кеш: ціни маршрутів

`OrderedDict`: на початку — найдавніше використаний, у кінці — найсвіжіший. `move_to_end(key)` — «щойно використали», `popitem(last=False)` — викинути найдавніший.

**🔮 Прогноз:** кеш на 2 записи: `put(A)`, `put(B)`, `get(A)`, `put(C)`. Хто залишиться?

<details>
<summary>Відповідь</summary>

`A` і `C`. `get(A)` зробив `A` найсвіжішим, тож найдавніше використаним став `B` — його й викинули. Без `get(A)` викинули б `A`.

</details>

### Вправа 6. `get` і `put`

- `get(key, default=None)`: немає — `misses += 1`, повернути `default`; є — `hits += 1`, `move_to_end`, повернути значення;
- `put(key, value)`: записати (існуючий ключ — теж у кінець); якщо записів більше за `capacity` — викинути найдавніший.

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self._data = OrderedDict()
        self.hits = 0
        self.misses = 0

    # YOUR CODE HERE
    # BEGIN SOLUTION
    def get(self, key, default=None):
        if key not in self._data:
            self.misses += 1
            return default
        self.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key, value):
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)
    # END SOLUTION

    def keys(self):
        return list(self._data)


cache = LRUCache(2)
cache.put("A", 1)
cache.put("B", 2)
assert cache.get("A") == 1
cache.put("C", 3)
print(cache.keys())
assert cache.keys() == ["A", "C"]
assert cache.get("B", "немає") == "немає"
assert (cache.hits, cache.misses) == (1, 1)
cache.put("A", 10)
assert cache.keys() == ["C", "A"] and cache.get("A") == 10
print("✅ Вправа 6 пройдена")

---
## 7. Пакет `dispatch` і його тести

Готовий пакет — у папці `dispatch_project` поруч з цим ноутбуком. У Colab ноутбук відкривається **без** файлів репозиторію, тому клітинка нижче в такому разі завантажує лише цю папку з GitHub (`git clone --sparse`, близько пів мегабайта).

In [ ]:
import os
import subprocess
import sys

PROJECT = "dispatch_project"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"
PATH_IN_REPO = "module_2/lessons/lesson_28_practicum_data_structures/dispatch_project"

if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
    subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", PATH_IN_REPO], check=True)
    PROJECT = os.path.join("course_repo", PATH_IN_REPO)

print("Проєкт:", PROJECT)
print(sorted(os.listdir(os.path.join(PROJECT, "dispatch"))))

In [ ]:
def run(*args):
    """Запустити команду Python у папці проєкту, як у терміналі."""
    result = subprocess.run([sys.executable, *args], cwd=PROJECT, capture_output=True, text=True)
    print((result.stdout + result.stderr)[-2500:])
    return result.returncode


run("-m", "dispatch")

Тести пакета — pytest (урок 25). Якщо pytest не встановлено, спершу: `pip install pytest` у терміналі (у Colab — клітинка `!pip install pytest`; зазвичай він уже є).

In [ ]:
code = run("-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no")
assert code == 0

### Вправа 7. Рівні дедлайни

Два термінові замовлення з дедлайном 09:30 — хто перший? Напиши функцію-ключ `urgent_key(order)` так, щоб купа видавала раніший дедлайн першим, а при рівному — менший номер. Перевіряємо на справжньому `Order` з пакета й на твоєму `MinHeap`.

In [ ]:
sys.path.insert(0, PROJECT)
from dispatch import Order


def urgent_key(order):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return (order.deadline, order.order_id)
    # END SOLUTION


heap = MinHeap(key=urgent_key)
for order in [Order(7, "Хорива, 5", "09:30"), Order(9, "Хрещатик, 1", "09:10"), Order(3, "Поділ", "09:30")]:
    heap.push(order)
result = [heap.pop().order_id for _ in range(3)]
print(result)
assert result == [9, 3, 7]
print("✅ Вправа 7 пройдена")

---
## Самоперевірка

1. Стек чи черга: скасування дій? обслуговування клієнтів у порядку дзвінків?
2. Чому `deque` для черги, а не `list`?
3. Що гарантує правило купи і чого не гарантує?
4. Навіщо в префіксному дереві `is_word`?
5. Який рядок робить кеш LRU, а не FIFO?

<details>
<summary>Відповіді</summary>

1. Скасування — стек (останнє першим); дзвінки — черга (перше першим).
2. `list.pop(0)` зсуває всі елементи — `O(n)`; `deque.popleft()` — `O(1)`.
3. Мінімум на вершині; повного порядку між гілками — ні.
4. Щоб відрізнити ціле слово від префікса («Хорива» — префікс, «Хорива, 5» — адреса).
5. `self._data.move_to_end(key)` у `get`.

</details>

## Далі

- Книга: [Урок 28](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_28/) — покрокові схеми кожної структури й архітектура пакета.
- Проєкт `dispatch_project/`: README з розділом «Спробуй зламати» — змінюй код і дивись, який тест падає.
- **Модуль 3** — бази даних; в уроці 30 Redis дасть ті самі черги, стеки, черги з пріоритетом і LRU-кеш — поза програмою.